Imports

In [123]:
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix

from pyts.approximation import SymbolicAggregateApproximation

from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit.circuit.library import zz_feature_map, pauli_feature_map



In [ ]:
EXPERIMENT_PARAMS = {
    # SAX configuration (fixed)
    "sax_window": 5,           # Window size for next-value prediction
    "sax_alphabet": 3,         # Fixed at 3 symbols (a, b, c)
    
    # Data generation
    "timeseries_points": 320,
    "gaussian_noise": 0.10,
    "data_seed": 123,
    
    # Split configuration
    "training_fraction": 0.8,
    "min_per_class_test": 3,
    "max_split_tries": 500,
    
    # Model configuration
    "repetitions_quantum": 1,  # Fixed at 1 for consistency
    "quantum_seed": 12345,
    
    # Next-value specific
    "step_symbol": 1,          # Predict 1 step ahead
    "sax_strategy": "uniform",
}


Data + SAX dataset

In [ ]:
import numpy as np
from pyts.approximation import SymbolicAggregateApproximation

def make_dataset_abc(
    *,
    window_size,
    alphabet_size,
    n_points,
    noise_std,
    seed,
    step_symbol=1,
    sax_strategy="uniform",
):
    """
    Generate next-value prediction dataset.
    
    Returns:
        X_sax: SAX word windows
        y: Next SAX symbol (targets)
    """
    rng = np.random.default_rng(seed)
    
    # Generate noisy sine wave
    t = np.linspace(0, 6 * np.pi, n_points)
    ts = np.sin(t) + rng.normal(0.0, noise_std, size=n_points)
    
    # Apply SAX encoding
    sax = SymbolicAggregateApproximation(n_bins=alphabet_size, strategy=sax_strategy)
    sax_seq = sax.fit_transform(ts.reshape(1, -1))[0]  # (n_points,)
    
    # Create windows -> next symbol mapping
    X_sax, y = [], []
    last_start = n_points - window_size - step_symbol
    for i in range(last_start):
        X_sax.append(sax_seq[i : i + window_size])
        y.append(sax_seq[i + window_size + step_symbol - 1])
    
    return np.asarray(X_sax), np.asarray(y)


Utilities (diagnostics, mapping, split)

In [ ]:
# Apply SAX encoding with fixed parameters
X_sax, y = make_dataset_abc(
    window_size=EXPERIMENT_PARAMS["sax_window"],
    alphabet_size=EXPERIMENT_PARAMS["sax_alphabet"],
    n_points=EXPERIMENT_PARAMS["timeseries_points"],
    noise_std=EXPERIMENT_PARAMS["gaussian_noise"],
    seed=EXPERIMENT_PARAMS["data_seed"],
    step_symbol=EXPERIMENT_PARAMS["step_symbol"],
    sax_strategy=EXPERIMENT_PARAMS["sax_strategy"],
)

def simple_angle_mapping(sax_matrix, margin=0.001):
    """
    Map SAX symbols to angles using straightforward linear distribution.
    
    Rationale: Simple monotonic spacing in [0, π] provides:
    - Numerical stability (no complex transformations)
    - Interpretability (direct geometric correspondence) 
    - Fairness (consistent encoding across all models)
    
    Args:
        sax_matrix: Symbolic SAX data
        margin: Small boundary offset
    
    Returns:
        angles_matrix: Numeric angle representation
        symbol_to_angle: Mapping dictionary
    """
    sax_as_strings = np.asarray(sax_matrix).astype(str)
    distinct_symbols = sorted(set(sax_as_strings.flatten()))
    symbol_count = len(distinct_symbols)
    
    if symbol_count < 2:
        raise ValueError(f"Insufficient symbol variety: {symbol_count}")
    
    # Evenly distribute angles in [margin, π - margin]
    angle_sequence = np.linspace(margin, np.pi - margin, symbol_count)
    symbol_to_angle = dict(zip(distinct_symbols, angle_sequence))
    
    # Transform symbols to angles
    angles_matrix = np.zeros(sax_as_strings.shape, dtype=float)
    for sym in distinct_symbols:
        angles_matrix[sax_as_strings == sym] = symbol_to_angle[sym]
    
    return angles_matrix, symbol_to_angle

def measure_sax_diversity(sax_matrix):
    """Calculate what fraction of SAX patterns are unique."""
    pattern_strings = [''.join(row.astype(str)) for row in np.asarray(sax_matrix)]
    return len(set(pattern_strings)) / max(1, len(pattern_strings))

def split_by_time(features, targets, train_fraction=0.8):
    """Temporal split preserving time series order."""
    total = len(features)
    cutoff = int(train_fraction * total)
    if cutoff <= 1 or cutoff >= total:
        raise ValueError("Invalid split boundaries")
    return features[:cutoff], features[cutoff:], targets[:cutoff], targets[cutoff:]

def prepare_labels(targets):
    """Encode trend labels for classification."""
    encoder = LabelEncoder()
    return encoder.fit_transform(targets), encoder

def display_results(model_tag, y_actual, y_predicted, label_encoder=None):
    """Show accuracy and confusion matrix for a model."""
    acc = accuracy_score(y_actual, y_predicted)
    print(f"{model_tag} accuracy={acc:.3f}")
    print(confusion_matrix(y_actual, y_predicted))
    if label_encoder is not None:
        print("classes:", list(label_encoder.classes_))
    return acc


Classical baselines (Linear + RBF)

In [ ]:
def train_rbf_classifier(train_features, test_features, train_targets, test_targets):
    """Train RBF kernel SVM with enhanced evaluation metrics."""
    
    # Build and train pipeline
    rbf_pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma="scale", probability=True))
    rbf_pipeline.fit(train_features, train_targets)
    
    # Predictions
    predictions = rbf_pipeline.predict(test_features)
    probabilities = rbf_pipeline.predict_proba(test_features)
    
    # Basic metrics
    acc = display_results("RBF-SVC", test_targets, predictions)
    macro_f1 = f1_score(test_targets, predictions, average='macro')
    
    # Calibration metric - original implementation
    pred_class = np.argmax(probabilities, axis=1)
    max_probs = np.max(probabilities, axis=1)
    is_correct = (pred_class == test_targets).astype(float)
    
    # Bin predictions by confidence level
    bin_count = 10
    calibration_error = 0.0
    for b in range(bin_count):
        lower = b / bin_count
        upper = (b + 1) / bin_count
        in_bucket = (max_probs > lower) & (max_probs <= upper)
        
        if np.sum(in_bucket) > 0:
            avg_conf = np.mean(max_probs[in_bucket])
            avg_acc = np.mean(is_correct[in_bucket])
            weight = np.sum(in_bucket) / len(is_correct)
            calibration_error += weight * abs(avg_conf - avg_acc)
    
    print(f"  Macro F1: {macro_f1:.3f}")
    print(f"  Calibration Error: {calibration_error:.3f}")
    
    return {
        "accuracy": acc,
        "f1_macro": macro_f1,
        "calibration_error": calibration_error,
        "predictions": predictions,
        "probabilities": probabilities
    }


Quantum kernel QSVC (reps = 1 or 2)

In [ ]:
def train_qsvc_with_analysis(train_features, test_features, train_targets, test_targets,
                               reps=1, random_state=12345):
    """Train QSVC with kernel eigenvalue analysis."""
    algorithm_globals.random_seed = random_state
    
    # Build quantum feature map and kernel
    zz_map = zz_feature_map(feature_dimension=train_features.shape[1], reps=reps, entanglement="full")
    sampler_instance = Sampler() 
    fidelity_calc = ComputeUncompute(sampler=sampler_instance)
    quantum_kernel = FidelityQuantumKernel(fidelity=fidelity_calc, feature_map=zz_map)
    
    # Train QSVC
    qsvc_model = QSVC(quantum_kernel=quantum_kernel)
    qsvc_model.fit(train_features, train_targets)
    
    # Predictions  
    predictions = qsvc_model.predict(test_features)
    acc = display_results(f"QSVC(reps={reps})", test_targets, predictions)
    macro_f1 = f1_score(test_targets, predictions, average='macro')
    
    # Kernel matrix analysis - compute effective dimensionality
    # Get kernel matrix for training data
    kernel_train_matrix = quantum_kernel.evaluate(x_vec=train_features)
    
    # Eigenvalue decomposition
    eigvals = np.linalg.eigvalsh(kernel_train_matrix)
    eigvals = np.sort(eigvals)[::-1]  # Descending
    eigvals = np.maximum(eigvals, 0)  # Remove numerical negatives
    
    # Compute effective rank (cumulative variance threshold at 99%)
    total_variance = np.sum(eigvals)
    if total_variance > 0:
        cumulative_var = np.cumsum(eigvals) / total_variance
        eff_rank = np.searchsorted(cumulative_var, 0.99) + 1
    else:
        eff_rank = 0
    
    print(f"  Macro F1: {macro_f1:.3f}")
    print(f"  Kernel Effective Rank: {eff_rank}/{len(eigvals)}")
    
    return {
        "accuracy": acc,
        "f1_macro": macro_f1,
        "effective_rank": eff_rank,
        "eigenvalues": eigvals,
        "predictions": predictions
    }


VQC and VQC with ZZ feature map (reps = 1 or 2)

In [ ]:
import numpy as np
from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit.primitives import StatevectorEstimator as Estimator

def train_vqc_classifier(train_features, test_features, train_targets, test_targets,
                         reps=1, random_state=12345, max_iterations=100):
    """Train Variational Quantum Classifier with standard metrics."""
    algorithm_globals.random_seed = random_state
    
    # Feature encoding and variational ansatz
    feature_encoder = zz_feature_map(feature_dimension=train_features.shape[1], reps=reps, entanglement="full")
    variational_circuit = real_amplitudes(num_qubits=train_features.shape[1], reps=2, entanglement="full")
    
    # Optimizer and sampler
    optimizer_instance = COBYLA(maxiter=max_iterations)
    sampler_instance = Sampler()
    
    # Build and train VQC
    vqc_model = VQC(feature_map=feature_encoder, ansatz=variational_circuit,
                    optimizer=optimizer_instance, sampler=sampler_instance)
    vqc_model.fit(train_features, train_targets)
    
    # Predictions
    predictions = vqc_model.predict(test_features)
    acc = display_results(f"VQC(fm_reps={reps})", test_targets, predictions)
    macro_f1 = f1_score(test_targets, predictions, average='macro')
    
    print(f"  Macro F1: {macro_f1:.3f}")
    
    return {
        "accuracy": acc,
        "f1_macro": macro_f1,
        "predictions": predictions
    }


In [ ]:
# Visualization functions for thesis-ready plots
import matplotlib.pyplot as plt

def plot_confusion_heatmap(conf_mat, model_label, class_labels=None):
    """Display confusion matrix as heatmap with annotations."""
    if class_labels is None:
        class_labels = ['Down', 'Flat', 'Up']
    
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(conf_mat, cmap='YlOrRd', interpolation='nearest')
    
    ax.set_xticks(range(len(class_labels)))
    ax.set_yticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, fontsize=11)
    ax.set_yticklabels(class_labels, fontsize=11)
    
    # Add values to cells
    for row in range(len(class_labels)):
        for col in range(len(class_labels)):
            value = conf_mat[row, col]
            color = 'white' if value > conf_mat.max()/2 else 'black'
            ax.text(col, row, str(value), ha='center', va='center', 
                   color=color, fontsize=13, weight='semibold')
    
    ax.set_xlabel('Predicted Class', fontsize=12, weight='bold')
    ax.set_ylabel('True Class', fontsize=12, weight='bold')
    ax.set_title(f'{model_label}: Confusion Matrix', fontsize=13, weight='bold', pad=15)
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Sample Count', rotation=270, labelpad=18, fontsize=10)
    
    plt.tight_layout()
    plt.show()


def plot_kernel_spectrum(eigval_array, model_label, eff_rank_value):
    """Visualize kernel eigenvalue decay and effective rank."""
    fig, ax = plt.subplots(figsize=(9, 5))
    
    indices = np.arange(len(eigval_array))
    ax.semilogy(indices, eigval_array, 'o-', linewidth=2, markersize=5, 
                alpha=0.75, label='Eigenvalues')
    
    ax.axvline(x=eff_rank_value-1, color='crimson', linestyle='--', 
              linewidth=2.5, label=f'Effective Rank = {eff_rank_value}', alpha=0.8)
    
    ax.set_xlabel('Eigenvalue Index', fontsize=12, weight='bold')
    ax.set_ylabel('Eigenvalue (log scale)', fontsize=12, weight='bold')
    ax.set_title(f'{model_label}: Kernel Eigenspectrum', fontsize=13, weight='bold', pad=15)
    ax.grid(True, alpha=0.25, which='both', linestyle=':')
    ax.legend(fontsize=11, loc='best')
    
    plt.tight_layout()
    plt.show()


def plot_calibration_reliability(proba_matrix, true_labels, model_label):
    """Create reliability diagram for calibration assessment."""
    # Compute calibration data
    pred_labels = np.argmax(proba_matrix, axis=1)
    max_confidences = np.max(proba_matrix, axis=1)
    correctness = (pred_labels == true_labels).astype(float)
    
    bin_count = 10
    bin_conf = []
    bin_acc = []
    bin_sizes = []
    
    for b in range(bin_count):
        low = b / bin_count
        high = (b + 1) / bin_count
        mask = (max_confidences > low) & (max_confidences <= high)
        
        if np.sum(mask) > 0:
            bin_conf.append(np.mean(max_confidences[mask]))
            bin_acc.append(np.mean(correctness[mask]))
            bin_sizes.append(np.sum(mask))
    
    if len(bin_conf) == 0:
        print(f'No calibration data for {model_label}')
        return
    
    # Compute calibration error
    total = len(true_labels)
    cal_error = sum((size/total) * abs(conf - acc) 
                    for conf, acc, size in zip(bin_conf, bin_acc, bin_sizes))
    
    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2.5, alpha=0.6, label='Perfect Calibration')
    
    sizes_scaled = [s * 8 for s in bin_sizes]
    ax.scatter(bin_conf, bin_acc, s=sizes_scaled, alpha=0.65, 
              edgecolors='navy', linewidths=2, c='skyblue',
              label=f'{model_label} (Error={cal_error:.3f})')
    
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_xlabel('Mean Predicted Confidence', fontsize=12, weight='bold')
    ax.set_ylabel('Empirical Accuracy', fontsize=12, weight='bold')
    ax.set_title(f'{model_label}: Reliability Diagram', fontsize=13, weight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.legend(fontsize=10, loc='upper left')
    
    plt.tight_layout()
    plt.show()

print('Visualization functions loaded')


In [ ]:
from collections import Counter
import numpy as np

# Execute complete experimental pipeline
print("="*70)
print("SAX NEXT-VALUE PREDICTION EXPERIMENT")
print("="*70)

# 1) Generate dataset with fixed parameters
X_sax, y = make_dataset_abc(
    window_size=EXPERIMENT_PARAMS["sax_window"],
    alphabet_size=EXPERIMENT_PARAMS["sax_alphabet"],
    n_points=EXPERIMENT_PARAMS["timeseries_points"],
    noise_std=EXPERIMENT_PARAMS["gaussian_noise"],
    seed=EXPERIMENT_PARAMS["data_seed"],
    step_symbol=EXPERIMENT_PARAMS["step_symbol"],
    sax_strategy=EXPERIMENT_PARAMS["sax_strategy"],
)

# 2) Dataset characteristics
symbol_distribution = Counter(y.tolist())
majority_class_fraction = max(symbol_distribution.values()) / len(y)
print(f"\nDataset Summary:")
print(f"  Window size: {EXPERIMENT_PARAMS['sax_window']}")
print(f"  Alphabet size: {EXPERIMENT_PARAMS['sax_alphabet']}")
print(f"  Total samples: {len(y)}")
print(f"  Symbol distribution: {dict(symbol_distribution)}")
print(f"  Majority baseline: {majority_class_fraction:.3f}")

# 3) Convert SAX to numeric features
X_angles, angle_map = simple_angle_mapping(X_sax)
diversity_score = measure_sax_diversity(X_sax)
print(f"  SAX pattern diversity: {diversity_score:.3f}")

# 4) Prepare data
y_numeric, label_enc = prepare_labels(y)
X_tr, X_te, y_tr, y_te = split_by_time(X_angles, y_numeric,
                                        train_fraction=EXPERIMENT_PARAMS["training_fraction"])
print(f"  Train/Test split: {len(y_tr)}/{len(y_te)}")

# 5) Train and evaluate all models
print("\n" + "="*70)
print("MODEL TRAINING AND EVALUATION")
print("="*70)

print("\n[1] RBF-SVC (Classical Baseline)")
print("-" * 50)
rbf_results = train_rbf_classifier(X_tr, X_te, y_tr, y_te)

print("\n[2] VQC (Variational Quantum Classifier)")
print("-" * 50)
vqc_results = train_vqc_classifier(X_tr, X_te, y_tr, y_te,
                                   reps=EXPERIMENT_PARAMS["repetitions_quantum"],
                                   random_state=EXPERIMENT_PARAMS["quantum_seed"],
                                   max_iterations=100)

print("\n[3] QSVC (Quantum Kernel SVM)")
print("-" * 50)
qsvc_results = train_qsvc_with_analysis(X_tr, X_te, y_tr, y_te,
                                        reps=EXPERIMENT_PARAMS["repetitions_quantum"],
                                        random_state=EXPERIMENT_PARAMS["quantum_seed"])

# 6) Comparative summary table
print("\n" + "="*70)
print("COMPARATIVE RESULTS SUMMARY")
print("="*70)
print(f"\n{'Model':<15} {'Accuracy':<12} {'Macro F1':<12} {'Calib. Error':<15} {'Kernel Rank':<12}")
print("-" * 70)
print(f"{'RBF-SVC':<15} {rbf_results['accuracy']:<12.3f} {rbf_results['f1_macro']:<12.3f} {rbf_results['calibration_error']:<15.3f} {'N/A':<12}")
print(f"{'VQC':<15} {vqc_results['accuracy']:<12.3f} {vqc_results['f1_macro']:<12.3f} {'N/A':<15} {'N/A':<12}")
print(f"{'QSVC':<15} {qsvc_results['accuracy']:<12.3f} {qsvc_results['f1_macro']:<12.3f} {'N/A':<15} {qsvc_results['effective_rank']:<12}")
print("="*70)

# Store results for further analysis
experimental_results = {
    "config": EXPERIMENT_PARAMS,
    "dataset_stats": {
        "n_samples": len(y),
        "symbol_distribution": dict(symbol_distribution),
        "pattern_diversity": diversity_score,
    },
    "rbf_svc": rbf_results,
    "vqc": vqc_results,
    "qsvc": qsvc_results,
}


## Section 8: Thesis-Ready Visualizations

In [ ]:
# Generate all thesis-ready plots
print("Generating visualizations for thesis...\n")

# Confusion matrices for all models
print("[1] Confusion Matrices")
print("-" * 50)

# Get confusion matrices
rbf_cm = confusion_matrix(y_te, rbf_results['predictions'])
vqc_cm = confusion_matrix(y_te, vqc_results['predictions'])
qsvc_cm = confusion_matrix(y_te, qsvc_results['predictions'])

# Plot them with consistent style
class_labels = [str(c) for c in label_enc.classes_]
plot_confusion_heatmap(rbf_cm, "RBF-SVC", class_labels)
plot_confusion_heatmap(vqc_cm, "VQC", class_labels)
plot_confusion_heatmap(qsvc_cm, "QSVC", class_labels)

# Calibration plot for RBF-SVC
print("\n[2] Calibration Reliability Diagram")
print("-" * 50)
plot_calibration_reliability(rbf_results['probabilities'], y_te, "RBF-SVC")

# Kernel spectrum for QSVC
print("\n[3] Kernel Eigenspectrum Analysis")
print("-" * 50)
plot_kernel_spectrum(qsvc_results['eigenvalues'], "QSVC", qsvc_results['effective_rank'])

print("\n✓ All visualizations generated successfully!")
